# Question 1B — Logistic Regression from scratch
**Objective:** binary classification (labels 0 and 1). Model logit $z_i=\mathbf{x}_i^T\mathbf{w}+b$ and probability $p_i=\sigma(z_i)=1/(1+e^{-z_i})$.

Binary cross-entropy: $J=-\frac{1}{m}\sum_i[y_i\log(p_i)+(1-y_i)\log(1-p_i)]$.

Stable implementation: $J=\frac{1}{m}\sum_i[\log(1+e^{z_i})-y_i z_i]$, with `np.logaddexp`.

By the chain rule, $\frac{\partial J}{\partial z_i}=p_i-y_i$, hence $\nabla_{\mathbf{w}}J=X^T(\mathbf{p}-\mathbf{y})/m$ and $\partial J/\partial b=\mathrm{mean}(\mathbf{p}-\mathbf{y})$. Gradient Descent updates are the same as for the linear model.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from src.q1_models import gradient_check
from sklearn.datasets import make_classification


## Model implementation using NumPy (not sklearn LogisticRegression)

In [2]:
class LogisticRegressionGD:
    """Minimize stable binary cross entropy: mean(logaddexp(0,z)-y*z)."""
    def __init__(self, learning_rate=0.13, epochs=2400):
        self.learning_rate, self.epochs = learning_rate, epochs
    @staticmethod
    def sigmoid(z):
        z=np.asarray(z,float)
        out=np.empty_like(z)
        mask=z>=0
        out[mask]=1/(1+np.exp(-z[mask]))
        ez=np.exp(z[~mask]);out[~mask]=ez/(1+ez)
        return out
    def _loss_grad(self,X,y,w,b):
        z=X @ w+b
        p=self.sigmoid(z)
        residual=p-y
        loss=float(np.mean(np.logaddexp(0,z)-y*z))
        return loss,X.T @ residual/len(y),float(np.mean(residual))
    def fit(self,X,y):
        X,y=np.asarray(X,float),np.asarray(y,float).ravel()
        if not set(np.unique(y)).issubset({0.,1.}):
            raise ValueError('Binary y values must be 0/1')
        self.weights_=np.zeros(X.shape[1]);self.bias_=0.
        self.loss_history_=[]
        for epoch in range(self.epochs):
            loss,dw,db=self._loss_grad(X,y,self.weights_,self.bias_)
            self.loss_history_.append(loss)
            self.weights_-=self.learning_rate*dw
            self.bias_-=self.learning_rate*db
        return self
    def predict_proba(self,X):
        return self.sigmoid(np.asarray(X) @ self.weights_ + self.bias_)
    def predict(self,X,threshold=0.5):
        return (self.predict_proba(X)>=threshold).astype(int)

## Stratified 80/20 split and train-only scaling

In [3]:
X,y=make_classification(n_samples=600,n_features=5,n_informative=4,n_redundant=0,
    weights=[.55,.45],class_sep=1.2,flip_y=.04,random_state=27)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
scaler=StandardScaler().fit(X_train)
X_train=scaler.transform(X_train);X_test=scaler.transform(X_test)
print('Training:',X_train.shape,'Test:',X_test.shape)

Training: (480, 5) Test: (120, 5)


In [4]:
model=LogisticRegressionGD(learning_rate=.13,epochs=2400).fit(X_train,y_train)
check=gradient_check(LogisticRegressionGD(),X_train[:20],y_train[:20])
print('Max finite-difference gradient error:',check)
assert check<1e-6
plt.figure(figsize=(8,4));plt.plot(model.loss_history_);plt.title('Logistic training loss');plt.xlabel('Epoch');plt.ylabel('Binary cross entropy');plt.grid(alpha=.2);plt.show()

Max finite-difference gradient error: 8.302217247013743e-11


/tmp/ipykernel_597/1318331123.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(8,4));plt.plot(model.loss_history_);plt.title('Logistic training loss');plt.xlabel('Epoch');plt.ylabel('Binary cross entropy');plt.grid(alpha=.2);plt.show()


## Held-out classification metrics
Positive class is **1**, decision threshold is **0.5**. ROC-AUC uses estimated probabilities instead of thresholded labels.

In [5]:
probs=model.predict_proba(X_test);pred=model.predict(X_test)
for label,score in [('Accuracy',accuracy_score(y_test,pred)),
    ('Precision',precision_score(y_test,pred)),('Recall',recall_score(y_test,pred)),
    ('F1',f1_score(y_test,pred)),('ROC-AUC',roc_auc_score(y_test,probs))]:
    print(f'{label}: {score:.4f}')
ConfusionMatrixDisplay.from_predictions(y_test,pred,cmap='Blues',colorbar=False)
plt.title('Logistic Regression held-out confusion matrix');plt.show()

Accuracy: 0.8833
Precision: 0.8333
Recall: 0.9259
F1: 0.8772
ROC-AUC: 0.9251


/tmp/ipykernel_597/3340558095.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Logistic Regression held-out confusion matrix');plt.show()


**Interpretation:** Unlike linear regression, the output is a probability between 0 and 1, and binary cross-entropy encourages calibrated probabilities. Accuracy should be considered together with class-sensitive metrics.